# Bayesian Spatial Synthetic Control in Python

## California's Proposition 99 with `scspill` and `mlsynth`

**Carlos Mendez** — Nagoya University (GSID)

Companion notebook for [the full tutorial](https://carlos-mendez.org/post/python_sc_bayes_spatial/).
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/starter-academic-v501/blob/master/content/post/python_sc_bayes_spatial/notebook.ipynb)

---

Three nested estimators on one panel, each relaxing one assumption of the last:

1. **Classical synthetic control** — donor weights on the simplex (Abadie, Diamond & Hainmueller 2010)
2. **Bayesian synthetic control** — the simplex replaced by a horseshoe prior
3. **Bayesian spatial synthetic control** — SUTVA on the donor pool dropped (Sakaguchi & Tagawa 2026)

The question the third stage can ask and the first two cannot: **who else was treated?**

This notebook follows the tutorial's own section numbering, so the two can be read
side by side. Every code block in the post appears here; the discussion does not —
go to the post for that.

> **Runtime.** As shipped, Stages 1–3 and the diagnostics run at the post's real
> budget (`m_iter = 500_000`), so the headline numbers match the published ones.
> The estimator survey in sections 13–15 runs at a reduced budget. Expect roughly
> **20–30 minutes** end to end on a free Colab CPU. Set `FAST = True` in section 5
> for a ~3-minute pass that reproduces the *shape* of every result — signs, ranks,
> orders of magnitude — but not the third decimal.

## 0. Environment setup (Colab)

Two libraries, both pinned. Run this once; if it asks you to restart, restart and
resume from the cell *after* the install.

In [ ]:
import subprocess
import sys


def pip(*spec):
    "Install into the running kernel. Returns pip's exit code."
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *spec]).returncode


# scspill 0.2.1 is the release every number in the post was produced under. The
# [numba] extra makes the samplers about five times faster and returns identical
# results on this panel -- but numba lags each new CPython release by months, so
# plain scspill is a correct fallback rather than a degraded one.
if pip("scspill[numba]==0.2.1"):
    print("No numba wheel for this Python; installing scspill without the extra.")
    pip("scspill==0.2.1")

# NOT "==1.0.0". The PyPI release numbered 1.0.0 is BEHIND git main at the same
# version string, so the pin has to be a commit. The [bayes] extra pulls numpyro,
# which BFSC, MVBBSC, BPSCS and SPOTSYNTH all import -- without it, four rows of
# the section 13 table fail with ModuleNotFoundError on an otherwise clean install.
pip("mlsynth[bayes] @ git+https://github.com/jgreathouse9/mlsynth.git"
    "@15f168bb90487098a7324be00b6663fcab0139ef")

print("done")

In [ ]:
# Version check and restart detection.
#
# Resolving the two installs above can swap numpy, scipy or pandas underneath a
# kernel that has already imported them, which produces confusing failures much
# later. Compare what is ON DISK against what is IN MEMORY and say so now.
#
# This cell deliberately imports none of them: section 5 pins the BLAS thread
# count, and that only works if it runs before numpy is first imported.
import importlib.metadata as md
import sys

need_restart = False
for p in ("numpy", "scipy", "pandas"):
    disk = md.version(p)
    live = sys.modules.get(p)
    if live is not None and getattr(live, "__version__", disk) != disk:
        need_restart = True
        print(f"{p:12s} {disk:12s} (in memory: {live.__version__})")
    else:
        print(f"{p:12s} {disk}")

for p in ("scspill", "mlsynth", "numba", "numpyro", "jax", "matplotlib"):
    try:
        print(f"{p:12s} {md.version(p)}")
    except md.PackageNotFoundError:
        print(f"{p:12s} not installed")

if need_restart:
    print("\n*** RESTART REQUIRED ***")
    print("Runtime > Restart session, then re-run from the NEXT cell.")
    print("Do NOT re-run the install cell.")
else:
    print("\nNo restart needed.")

## 1. Overview

California's 1988 Proposition 99 is the most replicated result in applied causal
inference. Its headline number rests on two assumptions that are rarely stated as
assumptions at all:

1. **The simplex** — donor weights must be non-negative and sum to one.
2. **SUTVA on the donor pool** — every donor state is untouched by California's policy.

The second is the interesting one. Proposition 99 raised the price of a pack in
California and did nothing to the price in Nevada, and in the classical fit Nevada
carries a weight of about 0.24.

**The data say the leak runs the other way.** Nevada's estimated spillover is
−5.50 packs per capita per year — its sales came in *below* its no-treatment path,
not above. So the contaminated classical estimate **understates** the effect.

### 1.1 Learning objectives

- **Distinguish** the two estimands a spillover-aware synthetic control reports, and
  see why classical synthetic control cannot express the second at all.
- **Implement** three nested estimators on one panel: `mlsynth.VanillaSC`,
  `mlsynth.BSCM` / `scspill` at zero spillover intensity, and `scspill.SCSPILL`.
- **Derive** the spillover-bias decomposition by hand on three donors.
- **Diagnose** the sampler with a prior predictive check, a Geweke joint distribution
  test and a prior-sensitivity grid.
- **Reconcile** two implementations of the same paper across six documented differences.

### 1.2 The road ahead

The three stages are nested — each keeps everything the previous stage assumed
except one restriction, which it replaces with something weaker. That is what makes
the comparison at the end meaningful: when the number moves, we know which
assumption moved it.

```
DiD                  every donor weighted 1/N; parallel trends
  |
Stage 1  Classical SC        weights chosen to fit, confined to the simplex
  |
Stage 2  Bayesian SC         simplex replaced by a horseshoe prior
  |
Stage 3  Bayesian spatial SC SUTVA on donors dropped; SAR layer, intensity rho
  |
  +--> two estimands: effect on California + spillover on each donor
```

Only the final stage can answer "who else was treated?", because only the final
stage has a parameter that represents the leak.

## 2. Key concepts

The vocabulary the rest of the notebook leans on. Two are slipperier than the rest:
*spillover bias*, which is what Stage 3 removes, and *effective sample size*, which
decides whether a credible interval means anything.

| # | Concept | In one line |
|---|---|---|
| 1 | **Potential outcomes under interference** $Y_{it}(d_1,\ldots,d_N)$ | the outcome unit $i$ takes under the whole assignment *vector*, not just its own |
| 2 | **ATT** | the effect averaged over post-treatment periods, for the unit that was treated |
| 3 | **Donor pool** | the 38 states with no large tobacco-control programme of their own |
| 4 | **Simplex constraint** $\alpha_j \ge 0$, $\sum_j \alpha_j = 1$ | forces the synthetic unit to interpolate rather than extrapolate |
| 5 | **Horseshoe prior** | continuous shrinkage: infinite spike at zero, Cauchy tails |
| 6 | **SUTVA** | one unit's treatment does not affect another's outcome |
| 7 | **SAR model** | each unit's outcome depends on a weighted average of its neighbours' |
| 8 | **Spillover effect** $\xi^{c}_{jt}$ | donor $j$'s observed outcome minus its no-treatment outcome |
| 9 | **Effective sample size** | how many *independent* draws an autocorrelated chain is worth |

## 3. The estimand, and two ways it goes wrong

Everything below serves one number: how many packs per capita did Proposition 99
cost California each year? That needs **a counterfactual** and **uncontaminated
donors**, and each can fail independently. Stages 1 and 2 answer the first
requirement. Stage 3 is the only one that addresses the second.

### 3.1 Potential outcomes when the treatment leaks

Index potential outcomes by the whole assignment vector $\mathbf{D}$ rather than by
$D_i$ alone — that is the notational commitment that lets us even state the problem:

$$Y_{it} = Y_{it}(\mathbf{D}), \qquad i = 1, \ldots, N, \qquad t = 1, \ldots, T$$

SUTVA is the restriction $Y_{it}(\mathbf{D}) = Y_{it}(D_i)$. With California as unit 1
and $\mathbf{e}_1$ the vector in which only California is treated, two estimands follow:

$$\xi_{0t} = Y_{1t}(\mathbf{e}_1) - Y_{1t}(\mathbf{0}), \qquad
\xi^{c}_{jt} = Y_{jt}(\mathbf{e}_1) - Y_{jt}(\mathbf{0})$$

Under SUTVA the second is *defined* to be zero — which is why classical synthetic
control cannot report it, cannot test it, and cannot be visibly wrong about it.

| Symbol | Meaning | In the code |
|---|---|---|
| $Y_{1t}$ | California's observed sales | `panel.df.query("state == 'California'").cigsale` |
| $Y_{1t}(\mathbf{0})$ | California's no-treatment sales | `result.counterfactual` |
| $\xi_{0t}$ | effect on California in year $t$ | `result.gap` |
| $\xi^{c}_{jt}$ | spillover onto donor $j$ in year $t$ | `result.spillover_panel[j][t]` |
| $T_0$ | last pre-treatment period (1987) | `result.inputs.T0` |

### 3.2 Why difference-in-differences will not do

$$\widehat{\mathrm{ATT}}_{\mathrm{DiD}} = \Big(\bar{Y}_{1,\mathrm{post}} - \bar{Y}_{1,\mathrm{pre}}\Big)
- \frac{1}{N-1}\sum_{j \neq 1} \Big(\bar{Y}_{j,\mathrm{post}} - \bar{Y}_{j,\mathrm{pre}}\Big)$$

Unbiased only under **parallel trends**. California starts below the donor average
and falls faster throughout the 1970s and 1980s, long before Proposition 99 exists —
the plot in section 6 makes this visible. Any method assuming California would
otherwise have tracked the average donor attributes a pre-existing trend to the policy.

### 3.3 The donor pool as a weighted average

Replace the equal weights $1/(N-1)$ with weights chosen so the blend tracks
California before treatment:

$$\widehat{Y}_{1t}(\mathbf{0}) = \sum_{j=2}^{N} \alpha_j Y_{jt} = \alpha^{\top} \mathbf{Y}^{c}_{t}$$

$$\widehat{\alpha} = \arg\min_{\alpha \in \Delta} \sum_{t=1}^{T_0}
\Big(Y_{1t} - \alpha^{\top}\mathbf{Y}^{c}_{t}\Big)^2, \qquad
\Delta = \Big\{\alpha : \alpha_j \geq 0, \, \sum_j \alpha_j = 1\Big\}$$

The set $\Delta$ is the simplex. Stage 1 solves this as written; Stage 2 replaces
$\alpha \in \Delta$ with a prior over all of $\mathbb{R}^{38}$; Stage 3 keeps Stage 2's
weights and changes what $\mathbf{Y}^{c}_{t}$ is assumed to *be*.

## 4. Three donors, four years, no computer

Before handing 38 donors to an optimiser, solve a version small enough to check by
hand. Everything in the next ten sections happens here first, in arithmetic you
could do on paper. These three cells need only numpy, so they run before the
library imports in section 5.

### 4.1 An exact blend on the simplex

In [ ]:
import numpy as np

A = np.array([10.0, 12.0, 14.0, 16.0])
B = np.array([20.0, 18.0, 16.0, 14.0])
C_donor = np.array([30.0, 30.0, 30.0, 30.0])
Z = np.array([15.0, 15.0, 15.0, 15.0])          # the treated unit, pre-treatment

blend = 0.5 * A + 0.5 * B + 0.0 * C_donor
print("blend      :", blend)
print("exact fit  :", np.allclose(blend, Z))
print("weights sum:", 0.5 + 0.5 + 0.0)

### 4.2 The treated unit outside the hull

Move the treated unit to 35 — above every donor. No convex combination can reach it,
so the simplex has to settle for the closest point it can construct.

In [ ]:
best_simplex = 0.0 * A + 0.0 * B + 1.0 * C_donor    # all weight on the highest donor
gap = np.array([35.0, 35.0, 35.0, 35.0]) - best_simplex
print("best simplex blend :", best_simplex)
print("pre-treatment RMSE :", np.sqrt((gap ** 2).mean()))

# Drop the sum-to-one requirement and the fit becomes exact.
alpha_unconstrained = np.array([0.0, 0.0, 7 / 6])
print("unconstrained blend:", alpha_unconstrained[2] * C_donor)
print("weights sum        :", alpha_unconstrained.sum().round(4))

### 4.3 What one leaky donor does

Donor B absorbs a spillover of −8 from a treatment it never received. The
identity to check is $\mathrm{bias} = -\sum_j \alpha_j \xi^{c}_j$.

In [ ]:
alpha_toy = np.array([0.5, 0.5, 0.0])

Y_no_treatment = np.array([18.0, 12.0, 30.0])   # A, B, C in year 5, absent any treatment
xi_toy = np.array([0.0, -8.0, 0.0])             # the spillover each donor absorbs
Y_observed = Y_no_treatment + xi_toy            # what we actually see

Y_treated_true = alpha_toy @ Y_no_treatment - 20.0  # the true post-treatment outcome

att_true = Y_treated_true - alpha_toy @ Y_no_treatment
att_naive = Y_treated_true - alpha_toy @ Y_observed

print(f"true ATT              : {att_true:+.1f}")
print(f"naive (SUTVA) ATT     : {att_naive:+.1f}")
print(f"bias                  : {att_naive - att_true:+.1f}")
print(f"-sum(alpha_j * xi_j)  : {-(alpha_toy * xi_toy).sum():+.1f}")

## 5. Setup: two libraries, two pins

Both packages are young and under active development, so both are pinned — installed
in section 0 above. The numbers here are reproducible only under these two pins.

| Package | Pin | Why |
|---|---|---|
| `scspill` | `0.2.1` (PyPI) | the release every number in the post was produced under |
| `mlsynth` | commit `15f168bb` | its PyPI release lags `main` by weeks at the same version string |

The `[numba]` extra on `scspill` is optional — about five times faster, identical
results. The `[bayes]` extra on `mlsynth` is not: four section-13 estimators import
`numpyro`.

**`FAST`** is the one dial in this notebook. Leave it `False` to reproduce the post.

In [ ]:
import os

# BLAS reduction order changes the last digits of every matrix product, and at
# N = 38 single-threaded BLAS is also faster than multi-threaded. Both are reasons
# to pin it, and it has to happen before numpy is imported.
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")
os.environ.setdefault("JAX_ENABLE_X64", "1")   # numpyro is float32 by default

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse.csgraph

import mlsynth
import scspill
from scspill import SCSPILL
from scspill.data import load_california

# ---------------------------------------------------------------------------
FAST = False        # True -> the whole notebook in ~3 minutes, shapes only
# ---------------------------------------------------------------------------

SEED = 20251022        # the R edition's seed, so the two are comparable
TREAT_YEAR = 1988

# The headline budget. Section 14 is the evidence that half a million draws for a
# 13-year effect is not excessive: the ATT is stable from about 100,000 onward,
# but rho needs roughly five times that before its ESS is reportable.
M_ITER, BURN = (4_000, 2_000) if FAST else (500_000, 250_000)

# Sections 13-15 fit many models, so they get a survey budget instead. At the
# headline budget SPILLSYNTH(sar) alone runs about 17 minutes.
SURVEY_ITER, SURVEY_BURN = (1_000, 500) if FAST else (4_000, 2_000)
NUTS_DRAWS = 250 if FAST else 500      # warmup = samples, for the numpyro estimators
BVSS_ITER = 100 if FAST else 300       # ~0.2 s per iteration; the post uses 1,000

# The R edition's published numbers, for comparison throughout.
R_EDITION = {"classical": -18.46, "horseshoe": -15.84, "sar": -16.59,
             "rho": 0.2226, "rho_ess": 2.93, "nevada": -3.75}

plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (9, 5),
                     "axes.grid": True, "grid.alpha": 0.3})

print(f"scspill {scspill.__version__}   mlsynth {mlsynth.__version__}")
print(f"FAST={FAST}   m_iter={M_ITER:,}   burn={BURN:,}")

## 6. The data

`scspill` ships the Proposition 99 panel and both spatial objects the third stage
needs, so there is nothing to download and nothing to merge.

### 6.1 The panel

In [ ]:
panel = load_california()
df = panel.df.copy()
donors = list(panel.spatial_W.index)

print(panel.description)
print(df.head())
print(f"\nshape {df.shape}   states {df['state'].nunique()}   "
      f"years {df['year'].min()}-{df['year'].max()}   missing {df.isna().sum().sum()}")

### 6.2 The spatial weights

- `spatial_w` — how exposed each **donor** is to the **treated** unit. One non-zero entry.
- `spatial_W` — donor-to-donor contiguity, row-normalised inside the estimator.

Under **rook contiguity** — the chess convention, in which two states are neighbours
only if they share a stretch of border, not merely a corner — Nevada is the only
donor-pool state bordering California. Oregon and Arizona border it too, but neither
is in the pool: both ran their own tobacco-control programmes. So there is **exactly
one leak channel**, and we can name it.

In [ ]:
# Pin the donor ordering once; every later section indexes off it.
W = panel.spatial_W.loc[donors, donors]      # donor-to-donor contiguity
w = panel.spatial_w.reindex(donors)          # each donor's exposure to California

print("California's neighbours inside the donor pool:")
print(w[w > 0])

print(W.iloc[:4, :4])
print(f"\nW is symmetric: {np.allclose(W.values, W.values.T)}")
deg = W.sum(axis=1)
print(f"degree: min {deg.min():.0f}  max {deg.max():.0f}  mean {deg.mean():.2f}")

### 6.3 Treatment in 1988, not 1989

Proposition 99 passed in November 1988; the tax took effect on 1 January 1989. ADH
and `mlsynth`'s own example use 1989. `scspill` uses **1988**, following the R
replication package. Neither is wrong, but they cannot be mixed — pin one so every
stage sees the same $T_0$.

In [ ]:
# Rebuild the treatment dummy from the stated rule and assert it matches, so a
# future change in the shipped column cannot silently move the post-period.
rebuilt = ((df["state"] == "California") & (df["year"] >= TREAT_YEAR)).astype(int)
assert (rebuilt.to_numpy() == df["treated"].to_numpy()).all()

years = np.sort(df["year"].unique())
wide = df.pivot(index="year", columns="state", values="cigsale")
y_treated = wide["California"].to_numpy()
post = years >= TREAT_YEAR
print(f"T0 = {(~post).sum()}   T1 = {post.sum()}   donors = {len(donors)}")

In [ ]:
fig, ax = plt.subplots()
for s in donors:
    ax.plot(years, wide[s], color="#8b9dc3", lw=0.7, alpha=0.5)
ax.plot(years, y_treated, color="#d97757", lw=2.6, label="California")
ax.axvline(TREAT_YEAR - 0.5, color="grey", ls="--")
ax.plot([], [], color="#8b9dc3", label=f"{len(donors)} donor states")
ax.set(xlabel="Year", ylabel="Cigarette sales (packs per capita)",
       title="California leaves the pack after 1988")
ax.legend()
plt.show()

## 7. Stage 1 — classical simplex synthetic control

Choose weights on the simplex so a blend of donors tracks California before 1988.

In [ ]:
common = dict(df=df, outcome="cigsale", treat="treated",
              unitid="state", time="year", display_graphs=False)

sc = mlsynth.VanillaSC(dict(common)).fit()

w_sc = pd.Series(sc.donor_weights).reindex(donors).fillna(0.0)
print(f"ATT                : {sc.att:.4f} packs per capita per year"
      f"   (R edition {R_EDITION['classical']:.2f})")
print(f"pre-treatment RMSE : {sc.pre_rmse:.4f}")
print(f"weights sum        : {w_sc.sum():.6f}")
print(f"active donors      : {(w_sc > 1e-4).sum()} of {len(donors)}\n")
print(w_sc[w_sc > 1e-4].sort_values(ascending=False).round(4).to_string())

Five of 38 donors carry the whole counterfactual and 33 get exactly zero. Two things
to notice: this reproduces the R edition to about 0.03 packs using a different
package and a different optimiser; and **Nevada is in the blend at about 0.24** —
the one state we have a specific reason to suspect of contamination is carrying a
quarter of the counterfactual.

In [ ]:
cf_sc = np.asarray(sc.time_series.counterfactual_outcome, float).ravel()

fig, (a1, a2) = plt.subplots(2, 1, sharex=True, figsize=(9, 7),
                             gridspec_kw={"height_ratios": [2, 1]})
a1.plot(years, y_treated, color="#6a9bcc", lw=2.2, label="California")
a1.plot(years, cf_sc, color="#00d4c8", lw=2, ls="--", label="Synthetic California")
a1.axvline(TREAT_YEAR - 0.5, color="grey", ls="--")
a1.legend()
a1.set(ylabel="Packs per capita", title="Stage 1: classical synthetic control")
a2.plot(years, y_treated - cf_sc, color="#d97757", lw=2)
a2.axhline(0, color="grey")
a2.axvline(TREAT_YEAR - 0.5, color="grey", ls="--")
a2.set(xlabel="Year", ylabel="Gap")
plt.show()

## 8. Stage 2 — Bayesian synthetic control

Replace the hard constraint with a **prior that prefers zero without forbidding
anything else**.

### 8.1 The horseshoe hierarchy

The horseshoe (Carvalho, Polson & Scott 2010) has an infinite spike at zero and
Cauchy tails:

$$\alpha_j \mid \lambda_j \sim \mathcal{N}\big(0, \lambda_j^2\big), \qquad
\lambda_j \mid \tau \sim \mathcal{C}^{+}(0, \tau), \qquad
\tau \sim \mathcal{C}^{+}(0, \sigma)$$

Most $\lambda_j$ come out tiny — shrinking that donor to nothing — while any single
one can be enormous if the likelihood insists. Nothing is forbidden; sparsity is
preferred rather than imposed.

### 8.2 Fitting it with `mlsynth.BSCM`

In [ ]:
bscm = mlsynth.BSCM({**common, "prior": "horseshoe", "n_iter": 20_000,
                     "burn_in": 10_000, "chains": 4, "seed": SEED}).fit()

w_bscm = pd.Series(bscm.donor_weights).reindex(donors).fillna(0.0)
beta0 = float(np.mean(np.asarray(bscm.posterior.beta0)))

print(f"ATT           : {bscm.att:.4f}")
print(f"95% CrI       : [{bscm.att_ci[0]:.4f}, {bscm.att_ci[1]:.4f}]")
print(f"intercept     : {beta0:.4f}   <- BSCM fits one; scspill does not")
print(f"weights sum   : {w_bscm.sum():.4f}   <- not 1, because of that intercept")
print(f"active donors : {(w_bscm.abs() > 0.01).sum()} of {len(donors)}")

### 8.3 The same model in `scspill`, at zero spillover intensity

Hold on to that intercept. `scspill` reports a Bayesian synthetic control about
**3 packs away** from this one, and the intercept is the entire explanation:

$$\text{BSCM:} \quad Y_{1t} = \beta_0 + \sum_j \alpha_j Y_{jt} + \varepsilon_t
\qquad\text{versus}\qquad
\text{scspill:} \quad Y_{1t} = \sum_j \alpha_j Y_{jt} + \varepsilon_t$$

Neither is more correct in the abstract. They answer different questions, and
averaging them would be meaningless.

**Setting $\rho = 0$ in the Stage 3 model returns Stage 2 exactly**, so the $\rho = 0$
ATT falls out of the *same* fit that section 9.5 runs. We therefore do that fit
**once, here**, and reuse the object for the rest of the notebook — which also saves
a second half-million-draw run.

In [ ]:
result = SCSPILL({
    **panel.config_kwargs(),      # df, columns, spatial_w, spatial_W, covariates
    "m_iter": M_ITER,
    "burn": BURN,
    "seed": SEED,
    "display_graphs": False,
    "max_effect_draws": 5_000,    # thin the effects sweep; the ATT is unaffected
}).fit()

print(f"fitted at m_iter={M_ITER:,}")

In [ ]:
from scspill.utils.scspill_helpers.sar.effects import treated_counterfactual

att_rho0 = result.effects_detail.att_scm          # the rho = 0 ATT
cf_rho0 = treated_counterfactual(result.inputs.Y0, result.inputs.Yc,
                                 result.inputs.Wn, result.inputs.wn,
                                 result.alpha_hat, rho=0.0)   # the whole path

print(f"scspill at rho = 0 : {att_rho0:.4f}")
print(f"mlsynth.BSCM       : {bscm.att:.4f}")
print(f"R edition Stage 2  : {R_EDITION['horseshoe']:.4f}")
print(f"rho=0 counterfactual, last 3 years: {cf_rho0[-3:].round(1)}")

## 9. Stage 3 — Bayesian spatial synthetic control

Everything so far has assumed the donors were bystanders. Stage 3 drops that.

### 9.1 Where the contamination enters

A synthetic control is built from *observed* donor outcomes, so if those already
contain a spillover, the counterfactual inherits it:

$$Y_{1t} - \sum_j \alpha_j Y_{jt} \, = \, \underbrace{\xi_{0t}}_{\text{what we want}}
\, - \, \underbrace{\sum_j \alpha_j \, \xi^{c}_{jt}}_{\text{bias}}$$

$$\mathrm{bias}_t = -\sum_j \alpha_j \, \xi^{c}_{jt}$$

This is the identity section 4.3 verified with $-(0.5 \times -8) = +4$. Three
consequences:

- If every $\xi^{c}_{jt} = 0$ the bias vanishes — SUTVA is the whole justification.
- The bias is a *product* of weight and spillover, so a contaminated zero-weight
  donor is harmless.
- Negative spillovers on positively-weighted donors push the estimate **toward zero**.

### 9.2 A spatial process on the donor outcomes

Each donor's outcome depends on its neighbours' outcomes *and* on the treated unit's,
with one intensity parameter $\rho$ governing both:

$$\mathbf{Y}^{c}_{t} = \rho \big( \mathbf{w} \, Y_{1t} + W \mathbf{Y}^{c}_{t} \big)
+ X_t \beta + \mathbf{u}_t, \qquad \mathbf{u}_t = \eta \gamma_t + \mathbf{e}_t$$

The error is not white noise: a latent AR(1) factor soaks up the common national
trends — surgeon-general reports, federal tax changes, the secular decline in smoking.

$$\gamma_t = \phi_\gamma \gamma_{t-1} + \epsilon_t, \qquad
\epsilon_t \sim \mathcal{N}(0, \sigma^2_\gamma), \qquad
\eta_{jk} \sim \mathcal{N}(0, \sigma^2_\eta \omega_k), \qquad
\omega_k \sim \mathcal{C}^{+}(0, 10)$$

Note that $\mathbf{w}$ multiplies $Y_{1t}$, California's **observed** outcome. Nevada's
residents respond to what California actually does, not to some counterfactual
California — which is what makes the system solvable.

| Symbol | Meaning | In the code |
|---|---|---|
| $\rho$ | spatial intensity, the leak | `result.rho_hat` |
| $\mathbf{w}$ | donor exposure to California | `panel.spatial_w` |
| $W$ | donor-to-donor contiguity | `panel.spatial_W` |
| $W_n$ | the same matrix, row-normalised | `result.inputs.Wn` |
| $\beta$ | covariate coefficients | `result.sar_posterior.beta` |
| $\gamma_t$, $\eta$ | latent AR(1) factor and loadings | `p_factors=1` |
| $\sigma^2$ | idiosyncratic variance | `result.sar_posterior.sigma2` |

### 9.3 Identification in closed form

The simplex is gone, and what replaces it as the identifying assumption is a
**perfect pre-treatment fit with unconstrained weights**:

$$\exists \, \alpha \in \mathbb{R}^{N} \, : \, Y_{1t}(\mathbf{0})
= \sum_j \alpha_j Y_{jt}(\mathbf{0}) \quad \text{for all } t$$

With $A = W + \mathbf{w}\alpha^{\top}$, the donors' no-treatment outcomes solve in
closed form,

$$\mathbf{Y}^{c}_{t}(\mathbf{0}) = \big(I_N - \rho A\big)^{-1}
\Big[\big(I_N - \rho W\big)\mathbf{Y}^{c}_{t} - \rho \, \mathbf{w} \, Y_{1t}\Big]$$

and **both** estimands follow immediately:

$$\xi_{0t} = Y_{1t} - \alpha^{\top}\mathbf{Y}^{c}_{t}(\mathbf{0}), \qquad
\boldsymbol{\xi}^{c}_{t} = \mathbf{Y}^{c}_{t} - \mathbf{Y}^{c}_{t}(\mathbf{0})$$

**Look at what is absent.** No $\beta$, no $\gamma_t$, no $\eta$, no $\sigma^2$. Only
$(\alpha, \rho, \mathbf{w}, W)$ and the observed data. That cancellation is why a
weakly identified nuisance block does not poison the effects.

### 9.4 The two-step sampler

The product $\rho \, \mathbf{w} \, \alpha^{\top}$ sits inside $\rho A$, so the exposure
channel identifies $\rho$ and $\alpha$ only jointly, and sampling them together mixes
badly. The paper's answer is a **cut posterior** — a deliberate refusal to let the
second step feed back into the first:

```
Step 1  horseshoe Gibbs on the pre-treatment regression      -> alpha
          |
        alpha fixed at its posterior mean
          |
Step 2  Gibbs sweep:  latent factors (FFBS) | beta (horseshoe)
                      sigma^2 (inverse gamma) | rho (adaptive RW Metropolis)
          |
        Effects in closed form: eigendecomposition + Sherman-Morrison
```

Three implementation details explain both the speed and the one weakness.

**The support for $\rho$.**

$$|\rho| < \frac{0.95}{\max\big(1, \, \max_i |\mu_i(W_n)|\big)}$$

For row-normalised contiguity the largest eigenvalue is exactly 1, so the
mathematical bound is $|\rho| < 1$; the 0.95 is a numerical safety margin the package
imposes, not a consequence of invertibility. Section 11.3 shows this is the *only*
prior setting that meaningfully moves the answer.

**The Jacobian.** Each Metropolis proposal needs $\log|I_N - \rho A|$ — an $O(N^3)$
determinant, half a million times. Pre-computing the eigenvalues $\mu_i$ of $A$ once
turns it into a sum ($\mu$ rather than $\lambda$, because $\lambda_j$ is already the
horseshoe's local scale in 8.1):

$$\log\big|I_N - \rho A\big| = \sum_{i=1}^{N} \log\big(1 - \rho \mu_i\big)$$

$O(N)$ per iteration instead of $O(N^3)$ — two minutes instead of two days.

**Adaptive step size.** Robbins–Monro tuning during burn-in, targeting 44%
acceptance (Gelman, Roberts and Gilks 1996):

$$\log s_{m+1} = \log s_m + (m+1)^{-0.6}\big(a_m - 0.44\big)$$

Adaptation stops at the end of burn-in, so the sampled portion is a genuine Markov
chain. Section 10 shows what happens when this is switched off — which is what the R
replication code does.

### 9.5 Fitting it

The fit already ran in section 8.3 — the configuration is the panel's own
`config_kwargs()` plus chain length, burn-in, seed, a plotting switch, and a cap on
how many posterior draws the effects sweep uses. Read the results off it.

In [ ]:
print(f"ATT         : {result.att:.4f}  95% CrI "
      f"[{result.att_ci[0]:.4f}, {result.att_ci[1]:.4f}]")
print(f"ATT at rho=0: {result.effects_detail.att_scm:.4f}   <- Stage 2, free")
print(f"rho         : {result.rho_hat:.4f}  95% CrI "
      f"[{result.rho_ci[0]:.4f}, {result.rho_ci[1]:.4f}]")
print(f"ESS(rho)    : {result.rho_ess:.1f}   acceptance {result.acc_rho:.3f}")
print(f"\nR edition   : ATT {R_EDITION['sar']:.2f}  rho {R_EDITION['rho']:.4f}  "
      f"ESS {R_EDITION['rho_ess']:.2f}")

**The credible interval for $\rho$ excludes zero.** That is the formal statement that
the data reject the restriction collapsing Stage 3 back to Stage 2: SUTVA on the
donor pool is not merely doubtful here, it is rejected by the model that nests it.

`scspill` ships its own diagnostics table. Read the `ess` column top to bottom.

In [ ]:
print(result.diagnostics(top_n_alpha=6).round(4))

$\sigma^2$ and the donor weights have effective sample sizes in the thousands. $\rho$
and $\beta$ have a fraction of that, **from the same chain** — the two quantities
leaning on the single contiguity channel are the hard ones, because the panel
contains one state's worth of evidence about how strongly policies leak.

Note the distinction that matters for how you report this: $\rho$'s posterior is
*tight* (a standard deviation of 0.043 on a support 1.9 wide). It is slow-mixing,
not weakly identified. The low ESS is autocorrelation from the random-walk
Metropolis step, not ignorance about $\rho$.

In [ ]:
result.plot(kind="panel")
plt.show()

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.5))
result.plot(kind="rho", ax=a)
result.plot(kind="trace", ax=b)
plt.tight_layout()
plt.show()

### 9.6 The spillover received by each donor

The second estimand is a whole panel — one spillover per donor per year. Sign
convention: `spillover_panel = Yc - Yc(0)`, so **negative** means the donor sold
*fewer* packs than it would have without Proposition 99.

In [ ]:
spill = result.spillover_panel.loc[TREAT_YEAR:]      # post-treatment rows only
means = spill.mean()
ranked = means.reindex(means.abs().sort_values(ascending=False).index)
print(ranked.head(6).round(4).to_string())
print(f"\nNevada absorbs {abs(ranked.iloc[0]) / abs(ranked.iloc[1]):.1f}x the next-largest donor.")
print(f"R edition: Nevada {R_EDITION['nevada']:.2f}")

The prior expectation was cross-border shopping *raising* Nevada's sales. The estimate
says they **fell**. Whatever mechanism dominates — advertising, media, social norms
crossing a border that tax arbitrage also crosses — the net effect on Nevada ran the
same way as the effect on California.

That direction has a consequence for the headline number, and it is the opposite of
what most people guess. Section 12 measures it.

## 10. Why these numbers differ from the R edition

The [R edition](https://carlos-mendez.org/post/r_sc_bayes_spatial/) reports the same
ATT to within a few tenths of a pack — and a 95% credible interval **0.38 packs
wide**, against the 12.7 packs the corrected configuration reports at the headline
budget.

### 10.1 Reproducing the R specification

`scspill` documents six departures from the authors' R replication code. Three have
escape hatches, so we can put the Python code back into the R specification and watch
what happens.

In [ ]:
R_SPEC = dict(beta_prior="ridge",        # departure 2: flat-plus-ridge, not horseshoe
              propagate_alpha=False,     # departure 3: alpha fixed at its posterior mean
              adapt_rho=False,           # departure 4: fixed Metropolis step
              step_rho=0.01)

rspec = SCSPILL({**panel.config_kwargs(), "m_iter": 5_000, "burn": 2_500,
                 "seed": SEED, "display_graphs": False, **R_SPEC}).fit()

rw = rspec.att_ci[1] - rspec.att_ci[0]
cw = result.att_ci[1] - result.att_ci[0]

print("R specification, at the R edition's own 5,000-iteration budget:")
print(f"   ATT      {rspec.att:+.4f}   (R edition {R_EDITION['sar']:+.2f})")
print(f"   rho      {rspec.rho_hat:.4f}    (R edition {R_EDITION['rho']:.4f})")
print(f"   ESS(rho) {rspec.rho_ess:.2f}      (R edition {R_EDITION['rho_ess']:.2f})")
print(f"\ninterval width: R spec {rw:.3f}  vs corrected {cw:.3f}  ({cw / rw:.0f}x wider)")

Independent code in a different language reproducing $\hat\rho$ to three decimals
**including the pathology** — an effective sample size of about 3 is the R sampler's
behaviour, faithfully reproduced.

Two distinct things were wrong with that interval, and they are worth keeping apart:

- **Effective sample size asks whether the interval is *reliable*** — whether the
  chain visited enough of the posterior for its quantiles to mean anything. At ESS 3, no.
- **`propagate_alpha` asks whether the interval is *complete*** — whether it accounts
  for everything the model is uncertain about. The R code varies $\rho$ while holding
  the donor weights fixed at their posterior mean, so the reported interval contains
  **no** uncertainty about which states make up synthetic California.

Running the R specification for a hundred times as many iterations does *not* widen
the interval. Propagating $\alpha$ does.

### 10.2 The six departures

`scspill` ships the full list as a dataframe — `pd.read_csv("scspill_departures.csv")`
in the post's repository, reproduced here since Colab has no local copy.

| # | Area | R replication code | `scspill` | Escape hatch | Changes the answer? |
|---|---|---|---|---|---|
| 1 | Covariates | scrambled by a $(T,N,K)$ vs $(N,T,K)$ memory-layout mismatch | a proper $(T,N,K)$ array throughout | drop covariates | **yes — the big one** |
| 2 | Prior on $\beta$ | flat-plus-ridge conditional | the paper's horseshoe | `beta_prior="ridge"` | modestly |
| 3 | ATT bands | vary $\rho$ only, $\alpha$ at its posterior mean | paired $(\alpha,\rho)$ draws | `propagate_alpha=False` | **yes — the interval** |
| 4 | $\rho$ sampler | fixed Metropolis step | Robbins–Monro toward 44% | `adapt_rho=False` | the interval, not the point |
| 5 | Factor scales | inconsistent $\omega_k$ conditionals | $\mathcal{N}(0,\sigma^2_\eta\omega_k)$, $\mathcal{C}^{+}(0,10)$ | none | little, but the sampler was invalid |
| 6 | FFBS initialisation | $\gamma_1$ inconsistent with its own conditionals | coherent $\gamma_0 = 0$ | none | little, but the sampler was invalid |

Departure 1 is the most instructive kind of bug. The covariate array was indexed as
though it were $(N,T,K)$ when it was laid out as $(T,N,K)$, silently shuffling which
state's price goes with which state's sales. Nothing crashes. Nothing looks wrong.
**A memory-layout mistake was doing a substantial share of the modelling.**

### 10.3 What a joint distribution test catches

Departures 5 and 6 were not found by staring at output. They were found by a
**Geweke joint distribution test** (section 11.2).

Draw parameters from the prior and simulate data from them — that samples the joint
distribution of parameters and data. Alternatively simulate data once and repeatedly
run one sweep of the Gibbs sampler: if every conditional is correct, this targets the
**same** joint distribution. Systematic disagreement means at least one conditional is
wrong.

Neither departure changes the California answer much. Both mean the R sampler was not
converging to any posterior at all. **A sampler with an incoherent conditional does
not announce itself** — it produces plausible numbers, converges, passes trace-plot
inspection, and is wrong.

## 11. Diagnostics

Three checks, all first-class functions in `scspill`, to run before believing any of
the numbers above. They take the model's *inputs* rather than the fitted result, so
pull those off the fit once.

In [ ]:
# result.inputs carries everything the sampler saw, already aligned.
Y0 = np.asarray(result.inputs.Y0, dtype=float).ravel()      # treated outcome, (T,)
Yc = np.asarray(result.inputs.Yc, dtype=float)              # donor outcomes, (T, N)
if Yc.shape[0] != Y0.size:
    Yc = Yc.T
X = None if result.inputs.X is None else np.asarray(result.inputs.X, dtype=float)

T0_idx = result.inputs.T0
Y0_pre, Yc_pre = Y0[:T0_idx], Yc[:T0_idx]
X_pre = None if X is None else X[:T0_idx]

print(f"Y0_pre {Y0_pre.shape}   Yc_pre {Yc_pre.shape}   "
      f"X_pre {None if X_pre is None else X_pre.shape}")

Note the shape of `X_pre`: a proper $(T_0, N, K)$ array. That third dimension is
departure 1 from section 10.2 — the R code indexed the same block as though it were
$(N, T_0, K)$.

### 11.1 Prior predictive check

Does the prior generate data that look anything like the data we have? If not, the
posterior is a fight between a badly-specified prior and the likelihood, and the
likelihood does not always win.

In [ ]:
from scspill.validation import prior_predictive

W_raw, w_raw = result.inputs.W_raw, result.inputs.w_raw

ppc = prior_predictive(Y0_pre, W_raw, w_raw, result.alpha_hat,
                       Yc_obs=Yc_pre, X=X_pre, p=0,
                       a0=3.0, b0=1.0, n_draws=2000, seed=SEED)

# PriorPredictiveResult carries `observed`, `stats` and `p_values` rather than a
# ready-made table; assemble the two that matter.
ppc_tab = pd.DataFrame({"statistic": list(ppc.p_values.keys()),
                        "observed": [ppc.observed[k] for k in ppc.p_values],
                        "p_value": list(ppc.p_values.values())})
print(ppc_tab.round(4).to_string(index=False))

Six of nine statistics land comfortably inside the prior predictive cloud. The three
that do not — the two autocorrelations and the common variance — are the same finding
twice over: the prior under-predicts persistence. Worth knowing, not fatal, and the
reason section 11.3 checks whether the priors move the answer.

### 11.2 The Geweke joint distribution test

In [ ]:
from scspill.validation import geweke_test

for m in ((5_000,) if FAST else (20_000, 200_000)):
    rep = geweke_test(kernel="simple", T0=4, N=4, K=0, p=1,
                      m_iid=m, m_mcmc=m, burn=5_000, seed=SEED)
    tab = pd.DataFrame(rep.table)
    # GewekeReport does the Bonferroni bookkeeping itself.
    print(f"m = {m:>7,}   max |z| = {tab['z'].abs().max():.2f}   "
          f"flagged {rep.n_flagged} of {len(tab)} at |z| > {rep.z_crit:.2f}   "
          f"passed = {rep.passed}")

### 11.3 Prior sensitivity

Vary the priors and ask whether the answer follows. The one setting that moves it is
the support for $\rho$ — which is section 9.4's point: the bound is a prior, not a
mathematical necessity.

In [ ]:
from scspill.validation import prior_sensitivity

grid = pd.DataFrame([
    dict(a0=1.0, b0=1.0, rho_lo=-0.99, rho_hi=0.99, step_rho=0.05),
    dict(a0=3.0, b0=1.0, rho_lo=-0.99, rho_hi=0.99, step_rho=0.05),
    dict(a0=0.1, b0=0.1, rho_lo=-0.99, rho_hi=0.99, step_rho=0.05),
    dict(a0=1.0, b0=1.0, rho_lo=-0.50, rho_hi=0.50, step_rho=0.05),   # truncated
    dict(a0=1.0, b0=1.0, rho_lo=-0.99, rho_hi=0.99, step_rho=0.01),
    dict(a0=5.0, b0=2.0, rho_lo=-0.99, rho_hi=0.99, step_rho=0.05),
])
sens = prior_sensitivity(Yc, W_raw, w_raw, result.alpha_hat, grid, X=X, p=1,
                         m_burn=1_000 if FAST else 5_000,
                         m_keep=2_000 if FAST else 20_000,
                         base_seed=SEED)

# prior_sensitivity returns a PriorSensitivityResult, not a dataframe. The long
# table -- 18 rows over six parameter labels -- is on `.table`. Only rho here.
print(sens.table.query("parameter == 'rho'")
               .drop(columns="parameter").round(4).to_string(index=False))

## 12. What the leak actually cost

Section 9.1 derived the bias of a SUTVA-imposing estimator as
$-\sum_j \alpha_j \xi^{c}_j$. The real panel supplies all the pieces, so check it
directly. (The post reads these from `stage2_alpha_posterior.csv` and
`stage3_spillover_effects.csv`; here they come straight off the live objects.)

In [ ]:
alpha = pd.Series(result.alpha_hat, index=donors)   # horseshoe posterior mean weights
xi = spill.mean()                                   # average post-treatment spillover

contrib = (alpha * xi).sort_values()
print(contrib.head(4).round(4).to_string())
print(f"\nsum_j alpha_j * xi_j             : {contrib.sum():+.4f}")
print(f"att (purged) - att_scm (contam.) : "
      f"{result.att - result.effects_detail.att_scm:+.4f}")

# The same plug-in on the SIMPLEX weights, where Nevada carries more.
print(f"\non simplex weights               : {(w_sc * xi).sum():+.4f}")
print(f"   Nevada weight: simplex {w_sc['Nevada']:.3f}  horseshoe {alpha['Nevada']:.3f}")

The identity holds to about 0.06 packs — the residual is because the purged ATT
averages over paired $(\alpha, \rho)$ draws while the plug-in uses $\hat\alpha$ alone.

**Nevada is 97% of the story**, from a weight of 0.200 times a spillover of −5.50.

**The direction is the one section 4.3 predicted.** Spillovers negative, weights
positive, so the bias is positive: the contaminated estimate is *closer to zero* than
the truth. The horseshoe estimate understates by 1.19 packs, about 7%. On the simplex
weights, where Nevada carries 0.242 rather than 0.200, it is 1.51 packs, about 8% —
**the classical estimate is the more contaminated of the two**, precisely because the
constraint pushed more weight onto the one leaking donor.

**And it is small.** A spatial model, half a million draws and a rejected SUTVA
assumption move the headline by 1.2 packs out of 17. The spillover was real,
statistically clear, and substantively modest **for California**. It was not modest
for Nevada — a different question, and the one classical synthetic control could not
have asked.

## 13. The rest of the catalogue

`mlsynth` ships 46 estimators. Eight more classes run on this panel — nine
configurations, since `SPILLSYNTH` has two methods worth separating. One column keeps
the table from lying: **`comparable`**. Several of these target a *different*
estimand, and reading them against the ladder would be the easiest way to draw a
false conclusion from a tidy-looking table.

Two of the eight need input the bare panel does not carry.

In [ ]:
# SpSyDiD wants the 39x39 UNIT-INCLUSIVE matrix, not the 38x38 donor block:
units = ["California"] + donors
W39 = pd.DataFrame(0.0, index=units, columns=units)
W39.loc[donors, donors] = W.values
W39.loc["California", donors] = w.to_numpy()
W39.loc[donors, "California"] = w.to_numpy()

# BPSCS wants point coordinates. Rather than hard-coding state centroids -- a second
# source of truth that could disagree with W -- embed the rook graph itself in 2-D by
# classical MDS on its shortest-path distances. These are NOT geographic coordinates,
# and the results table says so.
D_graph = scipy.sparse.csgraph.shortest_path(W39.to_numpy(), unweighted=True)
D_graph[~np.isfinite(D_graph)] = np.nanmax(D_graph[np.isfinite(D_graph)]) + 1.0

n = len(units)
J = np.eye(n) - np.ones((n, n)) / n          # the centring matrix
Bc = -0.5 * J @ (D_graph ** 2) @ J           # double-centred squared distances
ev, evec = np.linalg.eigh(Bc)
top2 = np.argsort(ev)[::-1][:2]
coords = pd.DataFrame(evec[:, top2] * np.sqrt(np.clip(ev[top2], 0, None)),
                      index=units, columns=["mds_1", "mds_2"])
df_coords = df.merge(coords.rename_axis("state").reset_index(), on="state", how="left")

print(coords.loc[["California", "Nevada", "Maine"]].round(3).to_string())
print(f"\nCA-NV distance {np.linalg.norm(coords.loc['California'] - coords.loc['Nevada']):.2f}"
      f"   CA-ME distance {np.linalg.norm(coords.loc['California'] - coords.loc['Maine']):.2f}")

The embedding has recovered the graph's coarse geography without ever being shown a
map. Now the survey itself — at `SURVEY_ITER`, not the headline budget, because
`SPILLSYNTH(sar)` at 500,000 draws alone takes about 17 minutes.

In [ ]:
SPECS = [
    ("BVSS", mlsynth.BVSS, dict(n_iter=BVSS_ITER, burn_in=BVSS_ITER // 2, seed=SEED),
     True, "bayesian", "ATT on the treated, weights shrunk toward the simplex"),
    ("MVBBSC", mlsynth.MVBBSC,
     dict(n_warmup=NUTS_DRAWS, n_samples=NUTS_DRAWS, n_chains=2, seed=SEED),
     True, "bayesian", "ATT on the treated, hard simplex, Bernstein-von Mises interval"),
    ("BFSC", mlsynth.BFSC,
     dict(n_factors=8, n_warmup=NUTS_DRAWS, n_samples=NUTS_DRAWS, n_chains=2, seed=SEED),
     True, "bayesian", "ATT via a latent-factor counterfactual"),
    ("BPSCS", mlsynth.BPSCS,
     dict(df=df_coords, covariates=["retprice"], coords=["mds_1", "mds_2"],
          n_warmup=NUTS_DRAWS, n_samples=NUTS_DRAWS, n_chains=2, seed=SEED),
     True, "spillover", "ATT under a distance-decay prior on neighbours"),
    ("SPILLSYNTH(sar)", mlsynth.SPILLSYNTH,
     dict(method="sar", spatial_W=W, spatial_w=w, p_factors=1,
          mcmc_iter=SURVEY_ITER, mcmc_burn=SURVEY_BURN, step_rho=0.01, mcmc_seed=SEED),
     True, "spillover", "the SAME paper, independently ported"),
    ("SPOTSYNTH", mlsynth.SPOTSYNTH,
     dict(selection="S1", forecast="loo", n_samples=2 * NUTS_DRAWS,
          n_warmup=NUTS_DRAWS, seed=SEED),
     True, "spillover", "ATT after screening contaminated donors out of the pool"),
    ("SPILLSYNTH(cd)", mlsynth.SPILLSYNTH, dict(method="cd", affected_units=["Nevada"]),
     False, "spillover", "measured against a DEMEANED leave-one-out baseline"),
    ("SpSyDiD", mlsynth.SpSyDiD, dict(spatial_matrix=W39),
     False, "spillover", "reports direct, total and average indirect effects at once"),
    ("ISCM", mlsynth.ISCM, dict(inference=True, n_draws=2000, random_state=SEED),
     False, "spillover", "imperfect-fit correction on its own normalisation"),
]

rows = []
for name, cls, kw, comparable, family, estimand in SPECS:
    try:
        r = cls({**common, **kw}).fit()
        rows.append(dict(estimator=name, family=family, att=round(float(r.att), 3),
                         comparable="yes" if comparable else "NO", estimand=estimand))
    except Exception as exc:          # a failure is information, not an error
        rows.append(dict(estimator=name, family=family, att=np.nan,
                         comparable="error",
                         estimand=f"{type(exc).__name__}: {str(exc).splitlines()[0][:60]}"))

print(pd.DataFrame(rows).to_string(index=False))

**The `comparable` column is doing real work.** The "NO" rows are not failures — they
are estimators answering different questions. `SPILLSYNTH(cd)` measures against its
*own* no-spillover baseline of about −10.5, not the simplex's −18.4. `SpSyDiD` reports
three effects at once (direct −17.11, total −20.16, average indirect **+14.88**);
quoting the first alone would be a choice, not a reading. `ISCM` returns an interval
spanning [−136, +61] — not wrong, just uninformative on 18 pre-treatment periods.

`SPILLSYNTH(method="sar")` is an **independent port of the same paper by a different
author**, sharing no code with `scspill`. Its agreement with Stage 3 is the strongest
external check either library gets.

At the headline budget, runtime across this table spans nearly five orders of
magnitude — 0.02 seconds for `SpSyDiD` against 994 for `SPILLSYNTH(sar)`. That is not
a quality ranking; it is the difference between a closed form and a half-million-draw
MCMC, and it is worth knowing before you put one in a bootstrap loop.

## 14. How long must the chain be?

Section 9.5 used 500,000 iterations to estimate a 13-year effect. That needs
justifying, and the justification is not "more is better." Watch the ATT settle early
while `ESS(rho)` keeps climbing — roughly linearly, at about 0.00055 effective draws
per kept draw.

In [ ]:
BUDGETS = (1_000, 2_000, 5_000) if FAST else (5_000, 20_000, 50_000, 100_000,
                                              250_000, 500_000)

for m in BUDGETS:
    r = SCSPILL({**panel.config_kwargs(), "m_iter": m, "burn": m // 2,
                 "seed": SEED, "display_graphs": False,
                 "max_effect_draws": 5_000}).fit()
    print(f"{m:>7,}  ATT {r.att:+.4f}  width {r.att_ci[1] - r.att_ci[0]:6.3f}  "
          f"rho {r.rho_hat:.4f}  ESS {r.rho_ess:7.2f}  acc {r.acc_rho:.3f}")

The ATT is stable from about 100,000 draws onward. `ESS(rho)` is not — and it is the
ESS, not the point estimate, that decides whether the interval means anything.
**Report the effective sample size beside every credible interval, or the interval is
decoration.**

## 15. Monte Carlo: does modelling the leak pay?

Everything so far has been one panel where the truth is unknown. `scspill` ships a
simulation module, so the same question can be asked where the truth is planted.

In [ ]:
from scspill.simulate import mc_grid

mc = mc_grid(Ns=(16,), T0s=(20,), T1=10,
             rhos=(-0.6, -0.3, -0.1, 0.0, 0.1, 0.3, 0.6),
             sims_per=10 if FAST else 60, K=1, beta=(1.0,), sigma2=0.1,
             treated=(0, 1, 2, 3),
             m_iter=1_000 if FAST else 3_000, burn=500 if FAST else 1_000,
             step_rho=0.05, seed=SEED)

print(mc.round(4).to_string(index=False))

The bias of the SUTVA-imposing estimator grows with $|\rho|$ and vanishes at
$\rho = 0$, which is what the identity in section 9.1 says it must do. Modelling the
leak costs nothing when there is no leak.

## 16. The whole ladder

Every number above, in one table. The post reads this from `att_ladder.csv`; here it
is rebuilt from the live objects. `r_edition` is the same panel estimated by the
authors' own R and C++ code, and `diff` is the honest measure of how much the
implementation choice mattered.

In [ ]:
ladder = pd.DataFrame([
    ("1.  Classical SC (simplex)",        float(sc.att),   R_EDITION["classical"]),
    ("2a. Bayesian SC (BSCM, intercept)", float(bscm.att), np.nan),
    ("2b. Bayesian SC (scspill, rho=0)",  float(result.effects_detail.att_scm),
                                          R_EDITION["horseshoe"]),
    ("3.  Bayesian spatial SC",           float(result.att), R_EDITION["sar"]),
], columns=["stage", "att", "r_edition"])
ladder["diff"] = ladder["att"] - ladder["r_edition"]

print(ladder.round(3).to_string(index=False))
print(f"\nspread: {ladder['att'].max() - ladder['att'].min():.2f} packs; "
      f"every stage agrees on the sign")

## 17. Which estimator should you choose?

Nothing here argues the spatial model is always right. It argues that the spatial
model answers a question the others cannot, and that you should know which question
you are asking before you pick.

```
Could the treatment have reached any donor unit?
|
+-- No, and you can defend it
|   |
|   +-- Is the treated unit inside the donors' convex hull?
|       |
|       +-- Yes ................ Classical SC     VanillaSC
|       |                        interpretable, sparse
|       +-- No / poor pre-fit .. Bayesian SC      BSCM or scspill at rho=0
|                                extrapolation allowed
|
+-- Yes, or you cannot rule it out
    |
    +-- Do you have a credible exposure structure (w and W)?
        |
        +-- Yes ................ Bayesian spatial SC   SCSPILL method="sar"
        |                        two estimands
        +-- No, but you can
            name the affected
            units .............. Screen or net out     SPOTSYNTH, ISCM,
                                                       SPILLSYNTH method="cd"
```

The first question is the one that gets skipped, and it is the only one with no
statistical answer. The data can tell you how large the leak was *given* that you
allowed for one; they cannot tell you to look.

The second is a diagnostic you already have — here the RMSE was 1.60 against a mean
of 117.7, so the simplex was not obviously straining. The third is the practical
constraint: someone must supply $\mathbf{w}$ and $W$, and those carry real content.
Contiguity is the natural default for a tax; trade flows, migration or commuting
intensity might be better elsewhere.

## 18. Discussion

**The effect on California is robust.** Every estimator on the ladder reports between
−15.7 and −18.8 packs per capita per year, and every interval on the ladder excludes
zero. Widening to the six comparable estimators of section 13 stretches the range to
−16.3 and −26.3 without ever changing the sign. If your interest is the headline
number, the classical estimate was fine.

**The composition of the donor pool is not robust at all.** The same data support
five active donors or 25, depending entirely on whether sparsity is imposed by a
constraint or expressed as a prior. Sentences like "synthetic California is mostly
Utah, Nevada, Montana and Connecticut" read like findings and are closer to artefacts
of $\Delta$.

**SUTVA is false here, and the direction is the surprise.** Nevada absorbed −5.50
packs per capita per year, an order of magnitude more than any other state, with a
credible interval for $\rho$ excluding zero. So the classical estimate was biased
*toward zero* — "fine, and slightly conservative, for a reason it could not have told
you about."

**What this does not establish.** The SAR layer makes nothing causal that was not
causal before. It models co-movement across a fixed, researcher-supplied graph;
swapping contiguity for another graph would produce different spillovers. The
identifying assumption in 9.3 is strong and untestable. And $\rho$ remains the
slowest-mixing parameter even at half a million draws: an ESS of 137 is reportable,
not comfortable.

## 19. Summary and next steps

- **Method.** Three nested estimators on one panel. Simplex → horseshoe → SAR layer.
  Each stage keeps everything the previous one assumed but one thing.
- **Data.** The ADH Proposition 99 panel, 39 states over 1970–2000, bundled with
  rook-contiguity weights in which Nevada is California's only donor-pool neighbour.
- **Result.** ATT −18.43 (simplex), −15.68 (horseshoe), −16.87 (spatial), with
  $\hat\rho = 0.316$ excluding zero and a Nevada spillover of −5.50 packs, 11 times
  the next-largest donor. Modelling the leak makes the estimated effect *larger*.
- **Inferential lesson.** The R edition reported a 95% interval 0.38 packs wide from a
  chain whose ESS for $\rho$ is 2.93. The corrected run reports 12.71 packs from an
  ESS of 137. Nothing about the policy changed.
- **Limitation.** $\rho$ is the slowest-mixing parameter, not the least identified one.
  Its posterior is tight; its chain is heavily autocorrelated.
- **Next step.** Everything here conditions on a contiguity graph nobody estimated.
  The natural follow-up is Bayesian estimation of the spatial weight matrix itself.

## 20. Exercises

1. **Move the treatment year.** Rebuild `df["treated"]` at 1989 rather than 1988 and
   rerun all three stages. Is the change larger or smaller than the gap between the
   simplex and the horseshoe?
2. **Break the graph on purpose.** Zero out Nevada's entry in `panel.spatial_w` so the
   model believes no donor borders California, and refit. What happens to $\hat\rho$,
   to the ATT, and to Idaho's and Utah's spillovers? This is the cleanest way to see
   how much of section 9's story rests on one entry in one vector.
3. **Change what "neighbour" means.** Replace rook contiguity with an inverse-distance
   or $k$-nearest-neighbours matrix built from state centroids, row-normalise, refit.
   Does the Nevada result survive?
4. **Buy a better $\rho$.** Section 14 shows ESS growing almost linearly at about
   0.00055 effective draws per kept draw. Extrapolate: how many iterations would an
   ESS of 400 take? Run it, check whether the linear rate holds that far out, and
   decide whether the answer changes anything you would report.
5. **A second case study.** `scspill.data.load_sudan()` ships the paper's other
   application — 34 African countries, 2000–2015, South Sudan's 2011 secession, with
   weights built from **bilateral trade** rather than borders. Dense where contiguity
   was sparse, so $\rho$ should be far better identified. Check whether it is.

## 21. References and further reading

1. Abadie, A., Diamond, A. and Hainmueller, J. (2010). Synthetic control methods for
   comparative case studies. *JASA*, 105(490), 493–505.
   [doi:10.1198/jasa.2009.ap08746](https://doi.org/10.1198/jasa.2009.ap08746)
2. Sakaguchi, S. and Tagawa, Y. (2026). Bayesian synthetic control with spillover
   effects. *The Econometrics Journal*.
   [doi:10.1093/ectj/utag006](https://doi.org/10.1093/ectj/utag006)
3. Carvalho, C. M., Polson, N. G. and Scott, J. G. (2010). The horseshoe estimator for
   sparse signals. *Biometrika*, 97(2), 465–480.
4. Vehtari, A. et al. (2021). Rank-normalization, folding, and localization: an
   improved $\widehat{R}$. *Bayesian Analysis*, 16(2), 667–718.
5. Gelman, A., Roberts, G. O. and Gilks, W. R. (1996). Efficient Metropolis jumping
   rules. *Bayesian Statistics 5*.

**Software**

- `scspill` — <https://quarcs-lab.github.io/scspill/>
- `mlsynth` — <https://mlsynth.readthedocs.io/>

**Where to go next**

- **[The full tutorial](https://carlos-mendez.org/post/python_sc_bayes_spatial/)** — every equation derived, the six departures in
  detail, the Monte Carlo study, and the evidence behind the 500,000-iteration budget.
- **[The R edition](https://carlos-mendez.org/post/r_sc_bayes_spatial/)** — the same
  three stages using the authors' own R and C++ code.
- **[The synthetic control ladder in Python](https://carlos-mendez.org/post/python_sc_dsc_sdid/)**
  — DiD through synthetic DiD, on the Brexit referendum.